# Практика работы со слоями в <br>PyTorch (torch.nn)

В этом ноутбуке мы разберем, как инициализировать и применять основные слои <br>нейронных сетей в PyTorch.
Мы сосредоточимся на:
1. Размерностях входных и выходных тензоров (это самая частая причина ошибок).
2. Синтаксисе создания слоев.
3. Важных нюансах (например, разница между режимами `train()` и `eval()`).

*Примечание: Мы не разбираем математическую суть слоев, предполагая,<br>что вы уже знакомы с ней.*

In [2]:
import torch
import torch.nn as nn

# Устройство для вычислений (CPU или GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Linear (Полносвязный слой)
Слой `nn.Linear` применяет линейное преобразование к входящим данным:

$y = xW^T + b$

Главное правило: размер ВЫхода первого слоя должен совпадать с размером <br>Входа у следующего.

**Основные параметры:**
* **in_feature**: int, количество признаков на входе в слой
* **out_feature**: int, количество признаков на выходе из слоя
* **bias**: bool, будет использоваться смещение или нет

In [3]:
# Создаем фиктивный батч данных: 32 объекта, у каждого 128 признаков
batch_size = 32
in_features = 128
x_linear = torch.randn(batch_size, in_features)
print(f"Входной тензор: {x_linear.shape}") # Ожидаем: [32, 128]

# 1. Создаем линейный слой
# Мы хотим получить 64 признака на выходе
linear_layer = nn.Linear(in_features=128, out_features=64)
print(f"\nЛинейный слой: {linear_layer}")

# Пропускаем данные через слой
out_linear = linear_layer(x_linear)
print(f"\nПосле Linear: {out_linear.shape}") # Ожидаем: [32, 64]


Входной тензор: torch.Size([32, 128])

Линейный слой: Linear(in_features=128, out_features=64, bias=True)

После Linear: torch.Size([32, 64])


In [4]:
# 2. Создаем линейный слой без bias
# Мы хотим получить 64 признака на выходе
linear_layer = nn.Linear(in_features=128, out_features=64, bias=False)
print(f"\nЛинейный слой: {linear_layer}")


Линейный слой: Linear(in_features=128, out_features=64, bias=False)


## 2. Функции активации

Нелинейные преобразования после слоёв нейронной сети.

In [5]:
# В PyTorch их можно вызывать как nn.Module (рекомендуется для построения классов)
relu = nn.ReLU()
out_relu = relu(out_linear)

print(f"После ReLU: {out_relu.shape}") # Размер не меняется: [32, 64]


После ReLU: torch.Size([32, 64])


Альтернатива: функциональный API (без создания объекта слоя, экономит память)

In [6]:
import torch.nn.functional as F

out_sigmoid = F.sigmoid(out_linear)

print(f"После Sigmoid: {out_sigmoid.shape}") # Размер не меняется: [32, 64]

После Sigmoid: torch.Size([32, 64])


## 3. Conv2d (Двумерная свертка)
Используется для работы с изображениями или 2D-картами признаков.

Важный момент: входной тензор для Conv2d в PyTorch всегда имеет размер <br>`(batch_size, channels, Height, Width)` или `(N, C, H, W)`.

**Основные параметры:**
* **in_channels**: int, количество каналов на входе в слой
* **out_channels**: int, количество каналов на выходе из слоя
* **kernel_size**: int, размер ядра свёртки
* **stride**: int, шаг смещения ядра
* **padding**: int, отступы у входного тензора
* **bias**: bool, будет использоваться смещение или нет

In [7]:
# Создаем батч из 16 цветных изображений 64x64 пикселя
batch_size = 16
channels_in = 3    # 3 канала у цветного изображения
height, width = 64, 64
x_conv = torch.randn(batch_size, channels_in, height, width)

# Создаем сверточный слой
# Мы хотим получить 32 канала на выходе, ядро 3x3.
# padding=1 нужен, чтобы сохранить пространственные размеры (H и W) после свертки 3x3
conv_layer = nn.Conv2d(
    in_channels=3,      # кол-во ВХОДных каналов
    out_channels=32,    # кол-во ВЫХОДных каналов
    kernel_size=3,      # размер ядра
    stride=1,           # шаг смещения
    padding=1           # отступы у входного тензора
)
print(f"Свёрточный слой: {conv_layer}")

out_conv = conv_layer(x_conv)
print(f"\nРазмер тензора До Conv2d:    {x_conv.shape}")   # [16, 3, 64, 64]
print(f"\nРазмер тензора После Conv2d: {out_conv.shape}") # [16, 32, 64, 64]


Свёрточный слой: Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

Размер тензора До Conv2d:    torch.Size([16, 3, 64, 64])

Размер тензора После Conv2d: torch.Size([16, 32, 64, 64])


In [10]:
# Пример без padding (размерность уменьшится)
conv_no_pad = nn.Conv2d(
    in_channels=3, out_channels=32,
    kernel_size=3,
    padding=0
)
print(f"Свёрточный слой: {conv_no_pad}")

out_conv_shrunk = conv_no_pad(x_conv)

print(f"\nРазмер тензора До Conv2d:  {x_conv.shape}")   # [16, 3, 64, 64]
print(f"\nПосле Conv2d без padding: {out_conv_shrunk.shape}") # [16, 32, 62, 62]

Свёрточный слой: Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))

Размер тензора До Conv2d:  torch.Size([16, 3, 64, 64])

После Conv2d без padding: torch.Size([16, 32, 62, 62])


In [12]:
# Уменьшение размера с помощью stride
conv_stride = nn.Conv2d(
    in_channels=3, out_channels=32,
    kernel_size=3,
    stride=2,
    padding=1
)
print(f"Свёрточный слой: {conv_stride}")

out_conv_shrunk = conv_stride(x_conv)

print(f"\nРазмер тензора До Conv2d:  {x_conv.shape}")   # [16, 3, 64, 64]
print(f"\nПосле Conv2d со stride=2: {out_conv_shrunk.shape}") # [16, 32, 62, 62]

Свёрточный слой: Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

Размер тензора До Conv2d:  torch.Size([16, 3, 64, 64])

После Conv2d со stride=2: torch.Size([16, 32, 32, 32])


In [14]:
# Создаем батч из 16 цветных изображений 224x224 пикселя
batch_size = 16
channels_in = 3    # 3 канала у цветного изображения
height, width = 224, 224
x_conv = torch.randn(batch_size, channels_in, height, width)

# Уменьшение размера с помощью stride
conv_stride = nn.Conv2d(
    in_channels=3, out_channels=32,
    kernel_size=3,
    stride=2,
    padding=1
)
print(f"Свёрточный слой: {conv_stride}")

out_conv = conv_stride(x_conv)
print(f"\nРазмер тензора До Conv2d:    {x_conv.shape}")   # [16, 3, 64, 64]
print(f"\nРазмер тензора После Conv2d: {out_conv.shape}") # [16, 32, 112, 112]

Свёрточный слой: Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

Размер тензора До Conv2d:    torch.Size([16, 3, 224, 224])

Размер тензора После Conv2d: torch.Size([16, 32, 112, 112])


## 4. Dropout и BatchNorm
Эти слои ведут себя по-разному в режиме обучения (`model.train()`) и в режиме <br>оценки (`model.eval()`).
Это критически важный момент для корректной работы <br>модели!

In [15]:
# --- DROPOUT ---
# Отключает нейроны во время обучения для борьбы с переобучением
dropout_layer = nn.Dropout(p=0.5)

x_dropout = torch.ones(2, 5) # Тензор из единиц для наглядности
print(f"Подали на вход Dropout:\n", x_dropout)

# В режиме обучения часть элементов занулится, остальные масштабируются на 1/p
dropout_layer.train()
print("\nDropout в режиме train:\n", dropout_layer(x_dropout))

# В режиме оценки (инференса) Dropout ничего не делает, просто пропускает данные как есть
dropout_layer.eval()
print("\nDropout в режиме eval:\n", dropout_layer(x_dropout))


Подали на вход Dropout:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

Dropout в режиме train:
 tensor([[2., 0., 2., 2., 2.],
        [2., 2., 2., 2., 0.]])

Dropout в режиме eval:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])


In [20]:
# --- BATCH NORM ---
# Нормализует данные по батчу. Для Conv2d используется BatchNorm2d
# ВАЖНО: num_features должен совпадать с количеством каналов (или признаков) НА ВХОДЕ в этот слой
# Обычно BatchNorm ставят ПОСЛЕ Conv2d, но ДО функции активации (или после,
# зависит от архитектуры, но num_features берется от выхода предыдущего слоя)

batch_size = 16
channels_in = 3    # 3 канала у цветного изображения
height, width = 64, 64
x_conv = torch.randn(batch_size, channels_in, height, width)

# Создаем сверточный слой
# Мы хотим получить 32 канала на выходе, ядро 3x3.
# padding=1 нужен, чтобы сохранить пространственные размеры (H и W) после свертки 3x3
conv_layer = nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3,
    stride=1,
    padding=1
)
print(f"Свёрточный слой: {conv_layer}")

out_conv = conv_layer(x_conv)
print(f"\nРазмер тензора До Conv2d:    {x_conv.shape}")  # [16, 3, 64, 64]
print(
    f"Размер тензора После Conv2d: {out_conv.shape}",    # [16, 32, 64, 64]
    f"mean = {out_conv.mean():.4f}",
    f"std = {out_conv.std():.4f}"
)

# Выход предыдущего Conv2d с 32 каналами
bn_layer = nn.BatchNorm2d(num_features=32)
print(f"\nСлой BatchNorm: {bn_layer}")

# Пропускаем наш тензор out_conv (у которого 32 канала) через нормализацию
out_bn = bn_layer(out_conv)
print(
    f"\nПосле BatchNorm2d: {out_bn.shape}",    # [16, 32, 64, 64]
    f"mean = {out_bn.mean():.4f}",
    f"std = {out_bn.std():.4f}"
)

Свёрточный слой: Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

Размер тензора До Conv2d:    torch.Size([16, 3, 64, 64])
Размер тензора После Conv2d: torch.Size([16, 32, 64, 64]) mean = -0.0342 std = 0.5732

Слой BatchNorm: BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)

После BatchNorm2d: torch.Size([16, 32, 64, 64]) mean = -0.0000 std = 1.0000


## 5. Transformer (Encoder)
Современный стандарт для работы с последовательностями.

В PyTorch есть готовые блоки `nn.TransformerEncoderLayer` и `nn.TransformerEncoder`.

⚠️ **Внимание к размерностям:** по умолчанию PyTorch ожидает формат <br>`(Seq_len, Batch_size, Feature_dim)`.
Но начиная с современных версий, <br>можно (и нужно для удобства) использовать аргумент `batch_first=True`, чтобы <br>работать с привычным `(Batch, Seq_len, Feature_dim)`.

In [23]:
# Создаем данные: Батч 8, длина последовательности 20 (например, 20 слов), размер эмбеддинга 64
batch_size = 8
seq_len = 20
d_model = 64
x_trans = torch.randn(batch_size, seq_len, d_model)
print(f"Входной тензор в блок Transformer: {x_trans.shape}\n") # [8, 20, 64]

# Создаем один слой Transformer Encoder
# nhead (количество голов внимания) должен быть делителем d_model без остатка! (64 / 8 = 8)
encoder_layer = nn.TransformerEncoderLayer(
    d_model=64,
    nhead=8,
    dim_feedforward=256, # Размер скрытого слоя внутри FFN
    batch_first=True,
    dropout=0.1
)
print(encoder_layer)

# Собираем блоки Transformer Encoder
transformer_encoder = nn.TransformerEncoder(
    encoder_layer=encoder_layer,
    num_layers=8 # Глубина модели (количество блоков)
)
print(f"\nБлоки Transformer Encoder:\n{transformer_encoder}")

out_trans = transformer_encoder(x_trans)
print(f"\nВыход Transformer Encoder: {out_trans.shape}") # [8, 20, 64] (размер не меняется)

Входной тензор в блок Transformer: torch.Size([8, 20, 64])

TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (linear1): Linear(in_features=64, out_features=256, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=256, out_features=64, bias=True)
  (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
)

Блоки Transformer Encoder:
TransformerEncoder(
  (layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (linear1): Linear(in_features=64, out_features=256, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)


## 6. Практический паттерн: Conv Block
На практике слои редко используются по одному.

Стандартный паттерн для сверточных сетей выглядит так:
<br>**Conv2d -> BatchNorm2d -> ReLU**

In [22]:
# Создаем классический блок
conv_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, padding=1),
    nn.BatchNorm2d(num_features=64),
    nn.ReLU(inplace=True), # inplace=True экономит память, перезаписывая тензор
)

# Проверяем на наших данных
x_sample = torch.randn(4, 3, 128, 128)
print(f"Вход в свёрточный слой:    {x_sample.shape}")  # [4, 3, 128, 128]

out_block = conv_block(x_sample)
print(f"Выход из свёрточного слоя: {out_block.shape}") # [4, 64, 64, 64]


Вход в свёрточный слой:    torch.Size([4, 3, 128, 128])
Выход из свёрточного слоя: torch.Size([4, 64, 128, 128])


## Итоги видео:
1. Всегда проверяйте `.shape` тензора после каждого нового слоя, особенно при переходе от Linear к Conv2d и наоборот.
2. Для Conv2d формат всегда `(Batch, Channels, Height, Width)`.
3. **Крайне важно** не забывайте переключать модель в `model.eval()` перед валидацией/тестированием.
4. В Transformer используйте `batch_first=True` для работы с привычными размерами `(Batch, Seq_len, Feature_dim)`.
